# Wave Lab 3 — convergence, vérification et crédibilité numérique

[🇫🇷 Français](../../docs/wave-convergence-verification-lab.md#fr) · [🇬🇧 English](../../docs/wave-convergence-verification-lab.md#en) · [🇪🇸 Español](../../docs/wave-convergence-verification-lab.md#es) · [🇵🇹 Português](../../docs/wave-convergence-verification-lab.md#pt)

**Question :** une simulation stable est-elle forcément correcte ? Non. Ici nous testons la convergence contre une solution exacte connue.

Chemin : `solution exacte → erreur → raffinement → ordre p → résolution par longueur d'onde → phase → énergie → budget d'erreurs`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (root / 'src').exists():
        sys.path.insert(0, str(root / 'src'))
        break

from diderot_mls.wave_verification import (
    convergence_study_1d, neumann_mode_exact_1d, neumann_mode_wavelength,
    normalized_l2_error, observed_orders, physical_energy_diagnostic_1d,
    solve_neumann_mode_1d,
)
from diderot_mls.waves_fdtd import initial_previous_1d, step_1d
from diderot_mls.von_neumann import phase_velocity_ratio_1d


## 1 — Une solution exacte avant le code
Nous choisissons `u(x,t)=cos(kx) cos(ckt)` avec `k=mπ/L`. Elle satisfait l'équation d'onde et les murs de Neumann. **Prédire avant d'exécuter :** la forme spatiale doit rester la même, seule son amplitude change de signe et de taille.

In [ ]:
x=np.linspace(0,1,401); c=1.0; m=3
times=[0.0,0.08,0.16,0.24]
plt.figure(figsize=(8,4))
for t in times:
    plt.plot(x,neumann_mode_exact_1d(x,t,c=c,m=m),label=f't={t:.2f}')
plt.xlabel('x'); plt.ylabel('u exact'); plt.title('Exp. 1 — mode exact de Neumann'); plt.legend(); plt.grid(alpha=.25); plt.show()


## 2 — Une grille : numérique contre exact
Une superposition visuelle n'est pas suffisante : nous calculons aussi une erreur L2 normalisée.

In [ ]:
T=.37
x,u,dt,r,nsteps=solve_neumann_mode_1d(80,final_time=T,c=1.0,m=3,target_courant=.7)
ue=neumann_mode_exact_1d(x,T,c=1.0,m=3)
err=normalized_l2_error(u,ue,x[1]-x[0])
print(f'dt={dt:.6g}, steps={nsteps}, Courant={r:.5f}, L2 error={err:.6e}')
plt.figure(figsize=(8,4)); plt.plot(x,ue,label='exact'); plt.plot(x,u,'--',label='numérique')
plt.xlabel('x'); plt.ylabel('u'); plt.title('Exp. 2 — numérique vs exact'); plt.legend(); plt.grid(alpha=.25); plt.show()


## 3 — Raffiner h, h/2, h/4, h/8
Le nombre de Courant reste voisin de 0.7 et `dt` est ajusté pour arriver exactement au même instant final. **Prédiction :** l'erreur doit décroître rapidement.

In [ ]:
rows=convergence_study_1d((40,80,160,320),final_time=.37,c=1.0,m=3,target_courant=.7)
print(' N     dx         dt         r       PPW       L2 error')
for row in rows:
    print(f'{row.intervals:3d}  {row.dx:9.6f} {row.dt:9.6f} {row.courant:7.4f} {row.points_per_wavelength:8.2f} {row.error_l2:12.5e}')
hs=np.array([r.dx for r in rows]); es=np.array([r.error_l2 for r in rows])
plt.figure(figsize=(6,4)); plt.loglog(hs,es,'o-',label='erreur mesurée')
ref=es[-1]*(hs/hs[-1])**2; plt.loglog(hs,ref,'--',label='pente h²')
plt.gca().invert_xaxis(); plt.xlabel('h = dx'); plt.ylabel('erreur L2'); plt.title('Exp. 3 — étude de raffinement'); plt.legend(); plt.grid(alpha=.25); plt.show()


## 4 — Reconstruire l'ordre observé
Si `E(h)≈C h^p`, alors `p≈log(Eh/Eh2)/log(h/h2)`. Pour un raffinement par 2, le dénominateur vaut `log(2)`. **Prédiction :** p doit tendre vers 2.

In [ ]:
orders=observed_orders(rows)
for coarse,fine,p in zip(rows[:-1],rows[1:],orders):
    print(f'{coarse.intervals:3d} -> {fine.intervals:3d} intervals : p = {p:.4f}')
plt.figure(figsize=(6,3.5)); plt.plot(range(1,len(orders)+1),orders,'o-'); plt.axhline(2.0,ls='--')
plt.xticks(range(1,len(orders)+1),[f'{rows[i].intervals}→{rows[i+1].intervals}' for i in range(len(orders))])
plt.ylabel('ordre observé p'); plt.title('Exp. 4 — l’ordre émerge du raffinement'); plt.grid(alpha=.25); plt.show()


## 5 — Stable mais grossier : points par longueur d'onde
Toutes les simulations ci-dessous respectent CFL. Nous augmentons seulement le nombre de points qui décrivent une longueur d'onde. **Prédiction :** stable partout, mais beaucoup plus précis quand la résolution augmente.

In [ ]:
m=8; T=.43; lam=neumann_mode_wavelength(m,1.0)
ppw=[]; errs=[]
for N in (32,64,128,256):
    x,u,dt,r,_=solve_neumann_mode_1d(N,final_time=T,c=1.0,m=m,target_courant=.7)
    ue=neumann_mode_exact_1d(x,T,c=1.0,m=m)
    ppw.append(lam/(x[1]-x[0])); errs.append(normalized_l2_error(u,ue,x[1]-x[0]))
    print(f'N={N:3d}, Courant={r:.4f}, points/λ={ppw[-1]:5.1f}, error={errs[-1]:.5e}')
plt.figure(figsize=(6,4)); plt.loglog(ppw,errs,'o-'); plt.xlabel('points par longueur d’onde'); plt.ylabel('erreur L2'); plt.title('Exp. 5 — stable ≠ suffisamment résolu'); plt.grid(alpha=.25); plt.show()


## 6 — Une petite erreur de phase s'accumule
Le Lab 2 donne la vitesse de phase numérique. Ici nous comparons le déphasage accumulé après plusieurs périodes pour plusieurs résolutions, toutes stables.

In [ ]:
m=8; L=1.0; k=m*np.pi/L; lam=2*L/m; r=.7
for N in (32,64,128,256):
    dx=L/N; theta=k*dx; ratio=float(phase_velocity_ratio_1d(r,np.array([theta]))[0])
    periods=10; phase_error=2*np.pi*periods*(ratio-1.0)
    print(f'N={N:3d}, PPW={lam/dx:5.1f}, v_phase_num/c={ratio:.8f}, Δphi after {periods} periods={phase_error:.4f} rad')


## 7 — Surveiller l'énergie
Pour l'équation continue idéale avec murs de Neumann, l'énergie est constante. Nous calculons ici une approximation de cette énergie physique. Ce diagnostic n'est pas l'invariant discret exact du leapfrog.

In [ ]:
N=160; x=np.linspace(0,1,N+1); dx=x[1]-x[0]; c=1.0; m=3; r=.7; dt=r*dx/c
u0=neumann_mode_exact_1d(x,0,c=c,m=m); up=initial_previous_1d(u0,c=c,dt=dt,dx=dx,boundary='neumann'); u=u0.copy()
energies=[]; times=[]
for n in range(700):
    un=step_1d(up,u,c=c,dt=dt,dx=dx,boundary='neumann')
    energies.append(physical_energy_diagnostic_1d(up,u,un,c=c,dt=dt,dx=dx)); times.append(n*dt)
    up,u=u,un
energies=np.array(energies); rel=(energies-energies[0])/energies[0]
print('max relative excursion =',float(np.max(np.abs(rel))))
plt.figure(figsize=(7,3.5)); plt.plot(times,rel); plt.xlabel('t'); plt.ylabel('(E-E0)/E0'); plt.title('Exp. 7 — diagnostic d’énergie'); plt.grid(alpha=.25); plt.show()


## 8 — Raffiner ne corrige pas un mauvais paramètre
La référence physique utilise `c=1.0`, mais le modèle numérique reçoit volontairement `c=1.02`. **Prédiction :** l'erreur numérique diminue d'abord, puis un plancher lié au mauvais paramètre apparaît.

In [ ]:
T=.70; c_true=1.0; c_model=1.02; m=3
hs=[]; wrong=[]; correct=[]
for N in (40,80,160,320):
    x,u_bad,_,_,_=solve_neumann_mode_1d(N,final_time=T,c=c_model,m=m,target_courant=.7)
    ref=neumann_mode_exact_1d(x,T,c=c_true,m=m); dx=x[1]-x[0]
    wrong.append(normalized_l2_error(u_bad,ref,dx))
    x,u_ok,_,_,_=solve_neumann_mode_1d(N,final_time=T,c=c_true,m=m,target_courant=.7)
    correct.append(normalized_l2_error(u_ok,ref,dx)); hs.append(dx)
    print(f'N={N:3d}: correct-c error={correct[-1]:.5e}, wrong-c error={wrong[-1]:.5e}')
plt.figure(figsize=(6,4)); plt.loglog(hs,correct,'o-',label='c correct'); plt.loglog(hs,wrong,'o-',label='c faux de +2%')
plt.gca().invert_xaxis(); plt.xlabel('h'); plt.ylabel('erreur L2'); plt.title('Exp. 8 — raffinement vs erreur de donnée/modèle'); plt.legend(); plt.grid(alpha=.25); plt.show()


## Fin du Lab 3
Nous avons maintenant séparé : **stabilité → convergence → précision de phase → propriétés physiques → erreur numérique / modèle / données**. Le passage futur à Saint-Venant pourra alors répondre à une vraie question de fidélité physique, et non simplement ajouter de la complexité.